<a href="https://colab.research.google.com/github/dhanwariagarvit21/unified_theory_of_orderflow_vol_implementation/blob/main/bivariate_hawkes_proces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import zipfile
import torch.nn.functional as F
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import expon
from scipy.optimize import minimize
from torch._higher_order_ops.associative_scan import associative_scan

In [ ]:
def clean_mem():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
columns = [
    "trade_id",
    "price",
    "qty",
    "quote_qty",
    "timestamp",
    "is_buyer_maker",
    "is_best_match"
]
df = pd.read_csv('/kaggle/input/datasets/garvitdhanwaria/btceur/BTCEUR-trades-2026-10.csv',header=None,names=columns)

In [ ]:
df["datetime"] = pd.to_datetime(
    df["timestamp"],
    unit="us",
    utc=True
)
df["direction"] = df["is_buyer_maker"].map({
    True: -1,
    False: 1
})
df["signed_qty"] = (
    df["direction"] * df["qty"]
)

df["signed_quote_qty"] = (
    df["direction"] * df["quote_qty"]
)
df["buy_event"] = (
    df["direction"] == 1
).astype(int)

df["sell_event"] = (
    df["direction"] == -1
).astype(int)
df = df.sort_values(
    ["timestamp", "trade_id"]
).reset_index(drop=True)

df["interarrival_ms"] = (
    df["timestamp"].diff() / 1000
)
df = df.sort_values(
    ["timestamp", "trade_id"]
).reset_index(drop=True)
df["interarrival_ms"] = (
    df["timestamp"].diff() / 1000
)

In [ ]:
print("Before cleaning:", len(df))

df = df.dropna(
    subset=[
        "timestamp",
        "price",
        "qty",
        "is_buyer_maker"
    ]
).copy()

df = df.sort_values(
    ["timestamp", "trade_id"]
).reset_index(drop=True)

print("After cleaning:", len(df))
print("Remaining missing values:")
print(df.isna().sum())

In [ ]:
buy_times = np.sort(
    df.loc[df["direction"] == 1, "timestamp"].unique()
)

sell_times = np.sort(
    df.loc[df["direction"] == -1, "timestamp"].unique()
)

In [ ]:
print("Buy events:", len(buy_times))
print("Sell events:", len(sell_times))

print("Buy sorted:", np.all(np.diff(buy_times) > 0))
print("Sell sorted:", np.all(np.diff(sell_times) > 0))

print("Buy range:", buy_times[0], buy_times[-1])
print("Sell range:", sell_times[0], sell_times[-1])

In [ ]:
t0 = min(buy_times[0], sell_times[0])
buy_t = (buy_times - t0) / 1_000_000
sell_t = (sell_times - t0) / 1_000_000

In [ ]:

def excitation_logcumsumexp(counts, times, beta, init_state):
    n = times.shape[0]
    mask = counts > 0
    m_long = mask.to(torch.long)
    idx = torch.cumsum(m_long, 0) - m_long
    exc = torch.zeros(n, dtype=times.dtype, device=times.device)
    if bool(mask.any()):
        t_act, c_act = times[mask], counts[mask]
        L = torch.logcumsumexp(beta * t_act + torch.log(c_act), dim=0)
        l_prev = L[idx.clamp(min=1) - 1]
        arg = l_prev - beta * times
        arg = torch.where(idx > 0, arg, torch.full_like(arg, -1e30))
        exc = torch.exp(arg)
    return exc + init_state * torch.exp(-beta * (times - times[0]))


def bivariate_hawkes_nll_gpu(buy_counts, sell_counts, params, times, T,
                             init_excitation_buy=0.0, init_excitation_sell=0.0):
    mu1, mu2, alpha_1, alpha_2, alpha_3, alpha_4, beta = params
    device, dtype = times.device, times.dtype

    def as_state(x):
        if torch.is_tensor(x):
            return x.to(device=device, dtype=dtype).reshape(())
        return torch.tensor(x, dtype=dtype, device=device)

    init_buy, init_sell = as_state(init_excitation_buy), as_state(init_excitation_sell)
    exc_buy = excitation_logcumsumexp(buy_counts, times, beta, init_buy)
    exc_sell = excitation_logcumsumexp(sell_counts, times, beta, init_sell)
    intensity_buy = mu1 + alpha_1 * exc_buy + alpha_2 * exc_sell
    intensity_sell = mu2 + alpha_3 * exc_buy + alpha_4 * exc_sell

    event_ll = torch.sum(buy_counts * torch.log(intensity_buy) + sell_counts * torch.log(intensity_sell))

    decay_integral = 1.0 - torch.exp(-beta * (T - times))
    buy_integral = torch.sum(buy_counts * decay_integral)
    sell_integral = torch.sum(sell_counts * decay_integral)
    init_integral = (1.0 - torch.exp(-beta * T)) / beta
    compensator = ((mu1 + mu2) * T
                   + (alpha_1 + alpha_3) / beta * buy_integral
                   + (alpha_2 + alpha_4) / beta * sell_integral
                   + (alpha_1 + alpha_3) * init_buy * init_integral
                   + (alpha_2 + alpha_4) * init_sell * init_integral)
    final_decay = torch.exp(-beta * (T - times[-1]))
    end_buy = (exc_buy[-1] + buy_counts[-1]) * final_decay
    end_sell = (exc_sell[-1] + sell_counts[-1]) * final_decay

    return -(event_ll - compensator), end_buy, end_sell


def check_excitation(buy_counts, sell_counts, times, beta=50.0, n_check=(10, 200, 2000), init=0.0):
    z = torch.tensor(float(init), dtype=times.dtype, device=times.device)
    b = torch.tensor(float(beta), dtype=times.dtype, device=times.device)
    local = times - times[0]
    eb = excitation_logcumsumexp(buy_counts, local, b, z)
    es = excitation_logcumsumexp(sell_counts, local, b, z)
    t, bc, sc = local.cpu().numpy(), buy_counts.cpu().numpy(), sell_counts.cpu().numpy()
    for n in n_check:
        d = np.exp(-beta * (t[n] - t[:n]))
        bb = (bc[:n] * d).sum() + init * np.exp(-beta * (t[n] - t[0]))
        ss = (sc[:n] * d).sum() + init * np.exp(-beta * (t[n] - t[0]))
        print(f"n={n}: fast=({eb[n].item():.8f}, {es[n].item():.8f})  brute=({bb:.8f}, {ss:.8f})")
        assert abs(eb[n].item() - bb) < 1e-7 and abs(es[n].item() - ss) < 1e-7, "excitation mismatch"
    print("excitation check passed")


def check_gradient(buy_counts, sell_counts, times, T):
    z = torch.tensor([-2.3, -2.7, 1.0, -3.0, 1.5, -3.0, 3.9], dtype=times.dtype, device=times.device, requires_grad=True)
    mu1, mu2 = torch.exp(z[0]), torch.exp(z[1])
    beta = torch.exp(z[6])
    a = torch.exp(z[2:6])
    nll, _, _ = bivariate_hawkes_nll_gpu(buy_counts, sell_counts, (mu1, mu2, a[0], a[1], a[2], a[3], beta),
                                         times - times[0], float((times[-1] - times[0]).item()))
    nll.backward()
    print("nll", nll.item(), "grad", z.grad.detach().cpu().numpy())
    assert torch.isfinite(z.grad).all(), "non-finite gradient"
    print("gradient check passed")

In [ ]:
all_times = np.concatenate([buy_t, sell_t])
all_directions = np.concatenate([
    np.ones(len(buy_t), dtype=np.int8),
    -np.ones(len(sell_t), dtype=np.int8)
])
order = np.argsort(all_times, kind="stable")

all_times = all_times[order]
all_directions = all_directions[order]
unique_times, inverse = np.unique(
    all_times,
    return_inverse=True
)

buy_counts = np.bincount(
    inverse,
    weights=(all_directions == 1).astype(float)
)

sell_counts = np.bincount(
    inverse,
    weights=(all_directions == -1).astype(float)
)

print("Unique event timestamps:", len(unique_times))
print("Buy events:", buy_counts.sum())
print("Sell events:", sell_counts.sum())
print("Simultaneous timestamps:",
      np.sum((buy_counts > 0) & (sell_counts > 0)))

In [ ]:
device = torch.device("cuda")
times_gpu = torch.as_tensor(
    unique_times,
    dtype=torch.float64,
    device=device
)

buy_counts_gpu = torch.as_tensor(
    buy_counts,
    dtype=torch.float64,
    device=device
)
sell_counts_gpu = torch.as_tensor(
    sell_counts,
    dtype=torch.float64,
    device=device
)
T = unique_times[-1]
T_gpu = float(T)

print("Unique timestamps:", len(unique_times))
print("Buy events:", buy_counts_gpu.sum().item())
print("Sell events:", sell_counts_gpu.sum().item())


In [ ]:
clean_mem()

In [ ]:

def fit_bivariate_hawkes_nll_gpu_batched(
    buy_counts,
    sell_counts,
    times,
    T,
    n_starts=12,
    batch_size=10000
):
    device = times.device
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")

    if n_starts < 1:
        raise ValueError("n_starts must be positive.")

    if len(times) == 0:
        raise ValueError("No observations supplied.")

    if len(buy_counts) != len(times) or len(sell_counts) != len(times):
        raise ValueError("Input arrays must have equal lengths.")

    if not torch.all(torch.isfinite(times)):
        raise ValueError("Timestamps contain non-finite values.")

    if not torch.all(times[1:] > times[:-1]):
        raise ValueError(
            "Timestamps must be strictly increasing. "
            "Aggregate simultaneous events first."
        )

    if T <= 0 or T < times[-1].item():
        raise ValueError(
            "T must be positive and at least as large as the final timestamp."
        )
    def transform_parameters(z):

        mu_buy = torch.exp(z[0])
        mu_sell = torch.exp(z[1])

        eta_buy = torch.sigmoid(z[2])
        split_buy = torch.sigmoid(z[3])

        eta_sell = torch.sigmoid(z[4])
        split_sell = torch.sigmoid(z[5])

        beta = torch.exp(z[6])

        alpha_bb = beta * eta_buy * split_buy
        alpha_bs = beta * eta_buy * (1 - split_buy)

        alpha_sb = beta * eta_sell * split_sell
        alpha_ss = beta * eta_sell * (1 - split_sell)

        return (
            mu_buy,
            mu_sell,
            alpha_bb,
            alpha_bs,
            alpha_sb,
            alpha_ss,
            beta
        )
    def objective_and_gradient(z_np):

        z = torch.tensor(
            z_np,
            dtype=torch.float64,
            device=device,
            requires_grad=True
        )

        params = transform_parameters(z)

        n_samples = len(times)
        carry_buy = torch.zeros(
            (),
            dtype=torch.float64,
            device=device
        )

        carry_sell = torch.zeros(
            (),
            dtype=torch.float64,
            device=device
        )
        total_loss = (
            (params[0] + params[1]) * times[0]
        )
        for i in range(0, n_samples, batch_size):

            end_idx = min(i + batch_size, n_samples)

            chunk_times = times[i:end_idx]
            chunk_buy = buy_counts[i:end_idx]
            chunk_sell = sell_counts[i:end_idx]

            chunk_T = chunk_times[-1] - chunk_times[0]

            local_times = (
                chunk_times - chunk_times[0]
            )

            chunk_loss, carry_buy, carry_sell = (
                bivariate_hawkes_nll_gpu(
                    chunk_buy,
                    chunk_sell,
                    params,
                    local_times,
                    chunk_T,
                    init_excitation_buy=carry_buy,
                    init_excitation_sell=carry_sell
                )
            )

            total_loss = total_loss + chunk_loss
            if end_idx < n_samples:

                gap = (
                    times[end_idx] - chunk_times[-1]
                )

                gap_decay_integral = (
                    1.0 - torch.exp(-params[6] * gap)
                ) / params[6]

                gap_integral = (
                    (params[0] + params[1]) * gap
                    + (
                        (params[2] + params[4]) * carry_buy
                        + (params[3] + params[5]) * carry_sell
                    ) * gap_decay_integral
                )

                total_loss = (
                    total_loss + gap_integral
                )

                # Decay excitation across the gap
                decay = torch.exp(
                    -params[6] * gap
                )

                carry_buy = carry_buy * decay
                carry_sell = carry_sell * decay
        tail = T - times[-1]

        tail_decay_integral = (
            1.0 - torch.exp(-params[6] * tail)
        ) / params[6]

        terminal_integral = (
            (params[2] + params[4]) * carry_buy
            + (params[3] + params[5]) * carry_sell
        ) * tail_decay_integral

        total_loss = (
            total_loss
            + (params[0] + params[1]) * tail
            + terminal_integral
        )
        if not torch.isfinite(total_loss):
            return (
                1e100,
                np.zeros(7, dtype=np.float64)
            )

        total_loss.backward()

        gradient = z.grad.detach().cpu().numpy()

        return (
            total_loss.item(),
            gradient.astype(np.float64)
        )
    n_buy = buy_counts.sum().item()
    n_sell = sell_counts.sum().item()

    mu_buy_initial = max(
        n_buy / T * 0.5,
        1e-8
    )

    mu_sell_initial = max(
        n_sell / T * 0.5,
        1e-8
    )

    beta_values = [
        1.0, 5.0, 10.0, 50.0, 100.0, 500.0
    ]

    eta_values = [
        0.2, 0.4, 0.6, 0.8
    ]

    split_values = [
        0.1, 0.3, 0.5, 0.7, 0.9
    ]

    starts = []
    for i in range(n_starts):

        beta0 = beta_values[i % len(beta_values)]

        eta_buy0 = eta_values[i % len(eta_values)]

        eta_sell0 = eta_values[
            (i // len(beta_values)) % len(eta_values)
        ]

        split_buy0 = split_values[
            i % len(split_values)
        ]

        split_sell0 = split_values[
            (i // len(split_values)) % len(split_values)
        ]

        starts.append(
            np.array([
                np.log(mu_buy_initial),
                np.log(mu_sell_initial),

                np.log(eta_buy0 / (1 - eta_buy0)),
                np.log(split_buy0 / (1 - split_buy0)),

                np.log(eta_sell0 / (1 - eta_sell0)),
                np.log(split_sell0 / (1 - split_sell0)),

                np.log(beta0)
            ], dtype=np.float64)
        )
    best_result = None
    all_results = []

    bounds = [
        (-20, 2),
        (-20, 2),
        (-10, 10),
        (-10, 10),
        (-10, 10),
        (-10, 10),
        (np.log(1e-3), np.log(1e4))
    ]

    for start_number, z0 in enumerate(starts, 1):

        result = minimize(
            objective_and_gradient,
            z0,
            method="L-BFGS-B",
            jac=True,
            bounds=bounds,
            options={
                "maxiter": 100,
                "ftol": 1e-8,
                "gtol": 1e-5,
                "maxls": 20
            }
        )

        all_results.append(result)

        print(
            f"Start {start_number}/{n_starts}: "
            f"NLL = {result.fun:.6f}, "
            f"Success = {result.success}, "
            f"Iterations = {result.nit}, "
            f"Gradient norm = {np.linalg.norm(result.jac):.6e}"
        )

        if (
            np.isfinite(result.fun)
            and np.all(np.isfinite(result.jac))
            and (
                best_result is None
                or result.fun < best_result.fun
            )
        ):
            best_result = result

    if best_result is None:
        raise RuntimeError(
            "All optimization runs failed."
        )
    z_best = torch.tensor(
        best_result.x,
        dtype=torch.float64,
        device=device
    )

    params = transform_parameters(z_best)

    (
        mu_buy,
        mu_sell,
        alpha_bb,
        alpha_bs,
        alpha_sb,
        alpha_ss,
        beta
    ) = [p.item() for p in params]
    branching_matrix = np.array([
        [
            alpha_bb / beta,
            alpha_bs / beta
        ],
        [
            alpha_sb / beta,
            alpha_ss / beta
        ]
    ])

    spectral_radius = max(
        abs(np.linalg.eigvals(branching_matrix))
    )
    return {
        "mu_buy": mu_buy,
        "mu_sell": mu_sell,

        "alpha_bb": alpha_bb,
        "alpha_bs": alpha_bs,
        "alpha_sb": alpha_sb,
        "alpha_ss": alpha_ss,

        "beta": beta,

        "branching_matrix": branching_matrix,
        "spectral_radius": spectral_radius,

        "log_likelihood": -best_result.fun,
        "negative_log_likelihood": best_result.fun,

        "success": best_result.success,
        "message": best_result.message,

        "gradient_norm": np.linalg.norm(best_result.jac),
        "iterations": best_result.nit,

        "all_results": all_results
    }


In [ ]:
bivariate_model = fit_bivariate_hawkes_nll_gpu_batched(
    buy_counts_gpu,
    sell_counts_gpu,
    times_gpu,
    T_gpu,
    n_starts=6,
    batch_size=10000
)
display(bivariate_model)

In [ ]:
clean_mem()

In [ ]:
bivariate_model = fit_bivariate_hawkes_nll_gpu_batched(
    buy_counts_gpu,
    sell_counts_gpu,
    times_gpu,
    T_gpu,
    n_starts=6,
    batch_size=15000
)
display(bivariate_model)

In [ ]:
clean_mem()

In [ ]:
bivariate_model = fit_bivariate_hawkes_nll_gpu_batched(
    buy_counts_gpu,
    sell_counts_gpu,
    times_gpu,
    T_gpu,
    n_starts=6,
    batch_size=5000
)
display(bivariate_model)